In [9]:
# ❶ 라이브러리 로드 & 원본 JSON 읽기
import json, pathlib, pprint, textwrap

SRC = pathlib.Path("/Users/taeyoonkwack/Documents/HCLT-KACL-2025/Korean_dialogue_Summarization/dataset/original/summarization_test.json")
DST = pathlib.Path("/Users/taeyoonkwack/Documents/HCLT-KACL-2025/Korean_dialogue_Summarization/dataset/finetuning_data/summarization_test.json")

with SRC.open(encoding="utf-8") as f:
    raw = json.load(f)

In [10]:
def convo_to_text(conv):
    """리스트→‘{speaker}: 발화’ 줄바꿈 문자열"""
    lines = [f"{t['speaker']}: {t['utterance']}" for t in conv]
    return "\n".join(lines)

SYS_PROMPT = (
    "당신은 대화 내용을 듣고 요약하는 assistant입니다. "
    "아래의 대화 내용을 참고하고, 주어진 keyword에 맞게 요약문을 작성하세요."
)

In [11]:
processed = []
for samp in raw:
    cid   = samp["id"]
    convo = convo_to_text(samp["input"]["conversation"])
    kw    = ", ".join(samp["input"]["subject_keyword"])
    user_msg = textwrap.dedent(f"""\
        [대화]
        {convo}

        [keyword] 
        {kw}

        위의 대화 내용을 keyword에 맞게 요약문을 작성하시오. 
        요약문을 제외한 그 무엇도 출력하지 마시오. 
        Return your summarization only — no additional text.""")
    
    processed.append({
        "id": cid,
        "messages": [
            {"role": "system",    "content": SYS_PROMPT},
            {"role": "user",      "content": user_msg},
            #{"role": "assistant", "content": samp["output"]}
        ],
        "output": "llm이 생성한 요약문",
    })

print("example =>"); pprint.pprint(processed[0], depth=3)


example =>
{'id': 'nikluge-2025-일상 대화 요약-test-000001',
 'messages': [{'content': '당신은 대화 내용을 듣고 요약하는 assistant입니다. 아래의 대화 내용을 참고하고, '
                          '주어진 keyword에 맞게 요약문을 작성하세요.',
               'role': 'system'},
              {'content': '        [대화]\n'
                          '        SD2000044: name1 님 그럼 제가 어~ 여름에 먹는 음식에 대해서 '
                          '한번 말씀드려볼게요.\n'
                          'SD2000044: 저는 여름에 팥빙수를 자주 먹거든요. 팥 팥빙수 진짜 맛있는 거 같아요. '
                          '제가 설빙도 자주 가고 어~ 아니면 어~ 그냥 그냥 평범한 팥빙수도 많이 먹는데 설빙에서 '
                          '제일 좋아하는 빙수는 초코 빙수예요. 초코 관련 돼 있으면 좀 다 좋더라고요.\n'
                          'SD2000044: 뭐 초코 딸기 빙수도 좋고 막 초코 오레오 이런 거도 좋고 제가 되게 '
                          '초코를 좀 많이 좋아하는 편이에요.\n'
                          'SD2000044: name1 님은 그럼 여름에 어~ 팥빙수 많이 안 드세요?\n'
                          'SD2000045: 어~ 저는 팥빙수보다는 과일을 더 자주 챙겨 먹는 스타일인 거 같아요.\n'
                          'SD2000045: 저는 항상 봄에서 여름 넘어갈 때쯤 이 되게 여름 과일이 되게 '
                 

In [12]:
with DST.open("w", encoding="utf-8") as f:
    json.dump(processed, f, ensure_ascii=False, indent=2)

print(f"✔️ Saved {len(processed):,} samples → {DST.resolve()}")

✔️ Saved 408 samples → /Users/taeyoonkwack/Documents/HCLT-KACL-2025/Korean_dialogue_Summarization/dataset/finetuning_data/summarization_test.json
